# Identifying topic using zero-shot modelling 

Going to generate some intial labels for my training dataset using zero shot modelling. 

In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import string 
from datasets import Dataset
import regex

from transformers import AutoModelForMaskedLM, pipeline, AutoTokenizer, AutoModel 
import torch
from sentence_transformers import SentenceTransformer

from torch.nn.functional import cosine_similarity

from sklearn.model_selection import GroupShuffleSplit

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cs = CommentsSaver(env='dev')
df = cs.read_all()

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [3]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
0,86496,Barnet,23/4301/FUL_14,23/4301/FUL,Bryn 15 Tenterden Grove London NW4 1SX,Objects,2024-02-28,"Dear Ms Bell,\n\n23/4301/FUL- conversion of tw...",2025-04-10,51.59042,-0.21855
1,86497,Barnet,23/4301/FUL_15,23/4301/FUL,15 Edgeworth Close London NW4 4HJ,Objects,2024-02-27,Dear Ms Bell\n\n23/4301/FUL- conversion of two...,2025-04-10,51.58520,-0.23718
2,86505,Westminster,23/05249/FULL_1,23/05249/FULL,None,Objects,2023-09-21,I am writing on behalf of the Knightsbridge Ne...,2025-04-10,NaN,NaN
3,86506,Westminster,23/05249/FULL_2,23/05249/FULL,None,Objects,2023-09-11,I object to the proposal to create a new two-s...,2025-04-10,NaN,NaN
4,86507,Westminster,23/05249/FULL_3,23/05249/FULL,None,Objects,2023-09-10,I object to this application because it will n...,2025-04-10,NaN,NaN


In [4]:
# Assume df is your DataFrame and 'application_id' is the column to group by
gss = GroupShuffleSplit(n_splits=1, train_size=0.05, random_state=42)

# Get the application_id groups
groups = df['application_id']

# Split the indices
train_inds, test_inds = next(gss.split(df, groups=groups))

# Create train and test DataFrames
df_train = df.iloc[train_inds].reset_index(drop=True)
#df_test = df.iloc[test_inds].reset_index(drop=True)

In [5]:
len(df_train)

1653

In [6]:
df_train.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
0,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN
1,86565,Southwark,22/AP/1782_2,22/AP/1782,None,Objects,2022-07-04,Required consultation has not happened. Neighb...,2025-04-10,NaN,NaN
2,86566,Southwark,22/AP/1782_3,22/AP/1782,None,Objects,2022-07-04,"New builds won't meet the requirement for ""Gre...",2025-04-10,NaN,NaN
3,86571,Barnet,22/4361/FUL_2,22/4361/FUL,46 Fitzjohn Avenue BARNET EN5 2HW,Objects,2022-09-29,The Barnet Society objects to this proposal.\n...,2025-04-10,51.65007,-0.20169
4,86570,Barnet,22/4361/FUL_1,22/4361/FUL,36 East View Barnet EN5 5TN,Objects,2022-09-30,Dear Sir/Madam\n\nApologies for just missing t...,2025-04-10,51.65680,-0.19791


In [7]:
df_train['cleaned_comment_text'] = df_train['comment_text'].astype(str)

In [8]:
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
nlp = NLP_Tasks(model_checkpoint)

# remove place names, numbers and non-ASCII characters
# df = nlp.remove_place_names(df=df, column='comment_text')
# df = nlp.remove_numbers(df=df, column='cleaned_comment_text')
df_train = nlp.remove_non_ascii(df=df_train, column='cleaned_comment_text')

Device set to use mps:0


In [ ]:
df_train = nlp.split_text_on_newline(df=df_train, column='cleaned_comment_text', filter_empty=True, filter_short=True, min_length=5)

In [10]:
len(df_train)

6936

In [11]:
df_train = nlp.split_text_on_period(df=df_train, column='cleaned_comment_text', filter_empty=True, filter_short=True, min_length=5)

In [12]:
len(df_train)

13338

In [13]:
df_train = nlp.remove_empty_rows(df=df_train, column='cleaned_comment_text')

In [14]:
len(df_train)

13329

In [15]:
df_train.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text
0,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,This is an objection to the second proposal wh...
1,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,the issue of security (flats are directly on t...
2,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,Note: this area of footpath is a dead end and ...
3,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,there is no fire route from the proposed flats...
4,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,"A comprehensive plan needs to be put in place,..."


In [16]:
topic_df = pd.read_excel('/Users/bea/Documents/AI4CI/projects/comment_summariser/list_of_topics.xlsx')

In [17]:
topic_list = set(topic_df['Topics'].astype(str).str.lower().tolist())

In [18]:
# strip whitespace
topic_list = [topic.strip() for topic in topic_list]

In [19]:
# remove 'nan' from list
topic_list = [topic for topic in topic_list if topic != 'nan']

In [20]:
topic_list.append('not found')

In [21]:
len(topic_list)

68

In [22]:
topic_list

['loss of property value',
 'applicants character',
 'against government policy',
 'impact on transport infrastructure',
 'replacement accommodation not suitable',
 'concerns about precarity',
 'area needs more hosuing',
 'impact on local amenities',
 'site for sale',
 'against local plan',
 'increased air pollution',
 'scale of development',
 'solar panels',
 'will improve safety',
 'prior planning decisions',
 'hazardous materials',
 'not meeting environmental standards',
 'contractual obligations',
 'lack of maintenance',
 'not affordable',
 'impact on visual amenity',
 'landscaping issues',
 'lack of space for social distancing',
 'private only spaces',
 'inadequate public consultation',
 'volume of objections',
 'density of development',
 'housing mix wrong',
 'increase in antisocial behaviour',
 'drainage and flooding',
 'increased smells',
 'in character with local area',
 'increased light pollution',
 'unaesthetic',
 'loss of parking',
 'increased noise pollution',
 'misleading

## Cosine similarity of embeddings to create intial labels

In [23]:
import torch
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

tqdm.pandas()

# Your model/tokenizer
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModel.from_pretrained(model_checkpoint)
model.eval()

# Helper function for getting sentence embedding
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
    return embeddings.squeeze().cpu().numpy()

# Get topic embeddings
topic_embeddings = []
for topic in topic_list:
    combined_text = " ".join(topic) if isinstance(topic, list) else topic
    emb = get_embedding(combined_text, model, tokenizer)
    topic_embeddings.append(emb)

# Convert to tensor matrix for cosine similarity calc
topic_embeddings_matrix = torch.tensor(topic_embeddings)

# Assign topic to each comment
def assign_topic(comment):
    emb = get_embedding(comment, model, tokenizer)
    similarities = cosine_similarity(
        emb.reshape(1, -1),
        topic_embeddings_matrix.numpy()
    )
    max_idx = similarities.argmax()
    # max_score = similarities[0, max_idx]
    return topic_list[max_idx]

# Apply to DataFrame
df_train['topic'] = df_train['cleaned_comment_text'].progress_apply(assign_topic)


/var/folders/4n/x6w1yfcx01qbymrsfpz4ybq00000gn/T/ipykernel_67470/3811296967.py:31: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1742908952465/work/torch/csrc/utils/tensor_new.cpp:257.)
  topic_embeddings_matrix = torch.tensor(topic_embeddings)
  0%|          | 0/13329 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 13329/13329 [08:05<00:00, 27.44it/s]


In [24]:
df_train

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,topic
0,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,This is an objection to the second proposal wh...,lack of space for social distancing
1,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,the issue of security (flats are directly on t...,developer previously built below standard dwel...
2,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,Note: this area of footpath is a dead end and ...,accomodation unsuitable for vulnerable people
3,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,there is no fire route from the proposed flats...,developer previously built below standard dwel...
4,86564,Southwark,22/AP/1782_1,22/AP/1782,None,Objects,2022-07-05,This is an objection to the second proposal wh...,2025-04-10,NaN,NaN,"A comprehensive plan needs to be put in place,...",developer previously built below standard dwel...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13333,86224,Barnet,24/2689/FUL_1,24/2689/FUL,"27, Elton Avenue, BARNET EN5 2EB",Objects,2024-07-16,The plan to split the property into 2 houses d...,2025-04-10,51.647693,-0.195469,These windows were NOT in original plans for t...,developer previously built below standard dwel...
13334,86224,Barnet,24/2689/FUL_1,24/2689/FUL,"27, Elton Avenue, BARNET EN5 2EB",Objects,2024-07-16,The plan to split the property into 2 houses d...,2025-04-10,51.647693,-0.195469,These windows need to be removed and the roofl...,developer previously built below standard dwel...
13335,86224,Barnet,24/2689/FUL_1,24/2689/FUL,"27, Elton Avenue, BARNET EN5 2EB",Objects,2024-07-16,The plan to split the property into 2 houses d...,2025-04-10,51.647693,-0.195469,Were the windows to be removed I can see no ob...,lack of space for social distancing
13336,86224,Barnet,24/2689/FUL_1,24/2689/FUL,"27, Elton Avenue, BARNET EN5 2EB",Objects,2024-07-16,The plan to split the property into 2 houses d...,2025-04-10,51.647693,-0.195469,Clearly this planning objection has prevented ...,accomodation unsuitable for vulnerable people


In [ ]:
# df_train.to_csv('../outputs/labelled_data/five_percent_labelled_data.csv', index=False)

In [26]:
import random

def generate_topic_metadata(topic_list):
    def random_hex_color():
        return "#{:06x}".format(random.randint(0, 0xFFFFFF))

    result = []
    for topic in topic_list:
        result.append({
            "text": topic,
            "suffix_key": topic[0].lower(),
            "background_color": random_hex_color(),
            "text_color": "#ffffff"
        })
    return result


In [27]:
topic_metadata = generate_topic_metadata(topic_list)

In [28]:
import json
with open("../outputs/labelled_data/topic_metadata.json", "w") as f:
    json.dump(topic_metadata, f, indent=4)

## Zero-shot classification to get us started 

In [29]:
# classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")

In [30]:
# # Define the candidate labels
# candidate_labels = topic_list

# # Get predictions for each comment
# predictions = []
# for text in cleaned_place_text:
#     result = classifier(text, candidate_labels, multi_label=False)
#     predictions.append(result)
# # Extract the labels and scores
# predicted_labels = []
# predicted_scores = []
# for result in predictions:
#     labels = result['labels']
#     scores = result['scores']
#     predicted_labels.append(labels)
#     predicted_scores.append(scores)
# # Create a DataFrame to store the results
# df_predictions = pd.DataFrame({
#     "comment_text": cleaned_place_text,
#     "predicted_label": predicted_labels,
#     "predicted_score": predicted_scores
# })

In [31]:
# # add column for the top label
# df_predictions['top_label'] = df_predictions['predicted_labels'].apply(lambda x: x[0])
# # add column for the top score
# df_predictions['top_score'] = df_predictions['predicted_scores'].apply(lambda x: x[0])

In [32]:
# # save the predictions to a csv file
# df_predictions.to_csv('../outputs/five_percent_zero_shot_label.csv', index=False)